# 🇻🇳 Piper TTS — Finetune cho tiếng Việt trên Kaggle (Fast)

Notebook tối ưu để **finetune Piper TTS** cho tiếng Việt nhanh nhất có thể trên Kaggle.

**Nguồn tham khảo (đã verify):**
- Code base: [OHF-Voice/piper1-gpl](https://github.com/OHF-Voice/piper1-gpl) v1.4.2
- Checkpoint warmstart: [rhasspy/piper-checkpoints](https://huggingface.co/datasets/rhasspy/piper-checkpoints) → `vi/vi_VN/vais1000/medium/epoch=4769-step=919580.ckpt` (846 MB, `global_step=919580`)
- Dataset mặc định: [doof-ferb/LSVSC](https://huggingface.co/datasets/doof-ferb/LSVSC) (100h, 24kHz, mở, multi-speaker → train như "voice trung bình")

## ⚙️ Cài đặt Kaggle (BẮT BUỘC)

1. **Accelerator**: chọn **GPU T4 x2** (Settings → Accelerator → GPU T4 x2). Gấp đôi tốc độ vs P100.
2. **Internet**: bật **Internet ON** (Settings → Internet).
3. **Persistence**: Settings → Persistence → **Files only** để giữ output giữa các session.
4. Nếu muốn dùng dataset gated (`capleaf/viVoice`, `thivux/phoaudiobook`) → thêm **Kaggle Secret** tên `HF_TOKEN`.

## 🚀 Các tối ưu tốc độ áp dụng

| Tối ưu | Cách | Tăng tốc |
|---|---|---|
| **Mixed precision** (`16-mixed` / `bf16-mixed`) | `--trainer.precision` | ~1.8x |
| **DDP 2 GPU** (T4 x2) | `--trainer.devices=2 --trainer.strategy=ddp_find_unused_parameters_true` | ~1.9x |
| **TF32 + cuDNN benchmark** | `--trainer.benchmark=true` + env | ~1.1x |
| **`pin_memory` + `persistent_workers`** | patch `dataset.py` (3 DataLoader) | ~1.2x |
| **Pre-cache `prepare_data()`** | gọi riêng cell, training subprocess re-iterate cache | tránh redo |
| **`num_workers` = 4** | match số CPU Kaggle | parallel I/O |
| **Sub-sample 3000 mẫu** | giảm thời gian/epoch | hội tụ nhanh khi warmstart cùng ngôn ngữ |
| **Warmstart vi_VN ckpt** | `--ckpt_path` resume + `+ADDITIONAL_STEPS` | hội tụ rất nhanh |
| **`max_steps` cap** | tránh chạy quá time-limit Kaggle 12h | dừng đúng lúc |

> ⚠️ `--ckpt_path` resume = restore cả `global_step`. Notebook tự đọc `global_step` từ base ckpt rồi cộng `ADDITIONAL_STEPS` để `max_steps` đúng nghĩa "train thêm N step".

> ⚠️ `ddp_find_unused_parameters_true` là **compatibility setting** cho manual-optimization GAN của VITS (không phải tăng tốc; chỉ để DDP không crash).


## 1. Kiểm tra môi trường (GPU / CPU / RAM)

In [ ]:
import os, sys, subprocess, shutil, multiprocessing
print("Python:", sys.version.split()[0])
print("CPU cores:", multiprocessing.cpu_count())
try:
    import torch
    print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}:", torch.cuda.get_device_name(i),
              f"({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")
except Exception as e:
    print("Torch chưa cài, sẽ cài ở cell sau:", e)

!free -h | head -2
!df -h /kaggle/working 2>/dev/null || df -h /tmp


## 2. Cài system + Python dependencies

Bao gồm: `build-essential`, `cmake`, `ninja-build`, `espeak-ng`, các Python deps cho training + dataset.


In [ ]:
%%bash
set -euo pipefail
# --- System packages ---
apt-get -qq update
apt-get -qq install -y build-essential cmake ninja-build espeak-ng libsndfile1 sox > /dev/null
echo "✅ apt deps installed"

# Verify espeak-ng có voice vi
espeak-ng --voices=vi || (echo "❌ espeak-ng không có voice vi!"; exit 1)


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
# Pin commit để reproducibility + tránh upstream bễ patch dataset.py (assert n==3 ở cell 7)
PIPER_COMMIT="2a60c2bd152356b613673ecaa0e15cbd3c0e502c"
if [ ! -d piper1-gpl ]; then
    git clone https://github.com/OHF-Voice/piper1-gpl.git
    cd piper1-gpl && git checkout "$PIPER_COMMIT"
else
    cd piper1-gpl
    git fetch --quiet origin "$PIPER_COMMIT" || true
    git checkout "$PIPER_COMMIT" 2>/dev/null || echo "⚠️  Could not checkout pinned commit; using existing."
fi

# Python deps — ép pin để khớp với piper1-gpl [train] extras + thêm deps notebook cần
pip install -q --upgrade pip
pip install -q \
    "lightning>=2,<3" "tensorboard>=2,<3" "tensorboardX>=2,<3" \
    "jsonargparse[signatures]>=4.27.7" "onnx>=1,<2" \
    "pysilero-vad>=2.1,<3" "cython>=3,<4" "librosa<1" \
    "scikit-build<1" "pathvalidate>=3,<4" "onnxruntime>=1,<2" \
    "datasets[audio]>=2.19,<4" "huggingface_hub>=0.23,<1" "soundfile>=0.12,<1"

# FIX: Kaggle preinstalls transformers mới (đòi huggingface_hub>=1.0) → conflict với pin hub<1 ở trên
# (datasets<4 bắt buộc hub<1). Piper train KHÔNG cần transformers — gỡ hẳn để
# torchmetrics (lightning dep) skip import bert_score (guard _TRANSFORMERS_GREATER_EQUAL_4_4).
pip uninstall -y transformers || true
# Verify đã gỡ thật (pip uninstall có thể fail silent)
python - <<'PYCHECK'
import importlib.util, pathlib, shutil
spec = importlib.util.find_spec("transformers")
if spec is not None and spec.origin:
    pkg_dir = pathlib.Path(spec.origin).parent
    print(f"pip uninstall failed -> rm -rf {pkg_dir}")
    shutil.rmtree(pkg_dir, ignore_errors=True)
    for di in pkg_dir.parent.glob("transformers-*.dist-info"):
        shutil.rmtree(di, ignore_errors=True)
import importlib; importlib.invalidate_caches()
assert importlib.util.find_spec("transformers") is None, "transformers van con!"
print("transformers removed OK")
PYCHECK

# Cài piper editable (build cython extension monotonic_align + libpiper espeakbridge)
pip install -q -e .

# Build in-place để import trực tiếp từ src/ luôn có piper.espeakbridge
python setup.py build_ext --inplace

# Build monotonic_align Cython core
bash build_monotonic_align.sh

# Verify
python -c "import piper; print('piper:', piper.__file__)"
python -c "import sys; sys.path.insert(0, '/kaggle/working/piper1-gpl/src'); import piper.espeakbridge; print('espeakbridge OK')"
python -c "from piper.train.vits.monotonic_align import maximum_path; print('monotonic_align OK')"
python -c "import datasets, soundfile, huggingface_hub; print('dataset deps OK')"
python -c "import lightning, torchmetrics, huggingface_hub as h; print(f'lightning {lightning.__version__} | torchmetrics {torchmetrics.__version__} | hf_hub {h.__version__} OK')"
echo "✅ piper1-gpl installed"


## 3. Patch `dataset.py` — bật `pin_memory` & `persistent_workers`

Hai tham số này **không có sẵn** trong code Piper. Patch tăng throughput ~20%.

Có assert đảm bảo đúng 3 DataLoader được patch (fail-fast nếu upstream đổi code).


In [ ]:
from pathlib import Path

ds = Path("/kaggle/working/piper1-gpl/src/piper/train/vits/dataset.py")
src = ds.read_text()

pattern = "            num_workers=self.num_workers,\n        )"
replacement = (
    "            num_workers=self.num_workers,\n"
    "            pin_memory=True,\n"
    "            persistent_workers=(self.num_workers > 0),\n"
    "        )"
)

if "pin_memory=True" not in src:
    n = src.count(pattern)
    assert n == 3, (
        f"Expected to patch 3 DataLoaders, found {n}. "
        f"Upstream dataset.py may have changed - inspect manually."
    )
    new_src = src.replace(pattern, replacement)
    compile(new_src, str(ds), "exec")  # syntax check
    ds.write_text(new_src)
    print(f"✅ Patched {n} DataLoaders with pin_memory + persistent_workers")
else:
    print("ℹ️  Already patched")

# -----------------------------------------------------------------------------
# Patch 2: DDP fix — prepare_data() chỉ chạy rank 0, nhưng setup() (mọi rank)
# cần self.piper_config (in-memory). Rank != 0 → assert chết → rank 0 treo NCCL.
# Fix: rank nào thiếu state thì tự chạy prepare_data() — idempotent: cache đầy đủ
# rồi (pre-cache cell + rank 0), mọi .exists() check hit → chỉ re-index, vài giây.
# Lightning có barrier giữa prepare_data và setup → không race.
# -----------------------------------------------------------------------------
src = ds.read_text()
setup_old = """    def setup(self, stage: str) -> None:
        assert self.piper_config is not None"""
setup_new = """    def setup(self, stage: str) -> None:
        if self.piper_config is None:
            # DDP: prepare_data() only runs on rank 0; rebuild state on other ranks.
            # Cache is already complete, so this is just re-indexing (fast).
            _LOGGER.info("piper_config missing (DDP rank != 0); rebuilding via prepare_data()")
            self.prepare_data()
        assert self.piper_config is not None"""

if "piper_config missing (DDP rank != 0)" in src:
    print("ℹ️  Patch 2 (DDP setup) already applied")
else:
    assert setup_old in src, "setup() anchor not found — upstream changed?"
    src = src.replace(setup_old, setup_new)
    compile(src, str(ds), "exec")
    ds.write_text(src)
    print("✅ Patch 2: setup() DDP-safe (rebuild piper_config on non-zero ranks)")

# -----------------------------------------------------------------------------
# Patch 3: train_dataloader thiếu shuffle=True (upstream bug) → SequentialSampler
# → data theo đúng thứ tự CSV mỗi epoch (DDP: DistributedSampler(shuffle=False)).
# Shuffle quan trọng cho convergence khi finetune.
# -----------------------------------------------------------------------------
src = ds.read_text()
train_old = """        return DataLoader(
            self.train_dataset,
            collate_fn="""
train_new = """        return DataLoader(
            self.train_dataset,
            shuffle=True,
            collate_fn="""

if "self.train_dataset,\n            shuffle=True," in src or "shuffle=True" in src:
    print("ℹ️  Patch 3 (shuffle) already applied")
else:
    assert src.count(train_old) == 1, "train_dataloader anchor not unique"
    src = src.replace(train_old, train_new)
    compile(src, str(ds), "exec")
    ds.write_text(src)
    print("✅ Patch 3: train_dataloader shuffle=True")

import subprocess
print(subprocess.check_output(["grep", "-n", "pin_memory|shuffle=True|rebuilding via prepare_data", "-E", str(ds)], text=True))


## 4. ⚙️ Cấu hình training — chỉnh các knob ở đây

Đây là cell duy nhất bạn cần chỉnh cho hầu hết case.


In [ ]:
# =============================================================================
# 🎛️  USER KNOBS — chỉnh ở đây
# =============================================================================

# --- Dataset (default: doof-ferb/LSVSC — open, standard HF datasets API, 24kHz, 100h) ---
# Các option khác:
#   "doof-ferb/LSVSC"        : 100h, mở, MIXED-SPEAKER (không có cột speaker thật).
#                              → train như single "voice trung bình". Tốt cho demo TTS tiếng Việt.
#   "capleaf/viVoice"        : 1016h, GATED (HF_TOKEN), có cột speaker → có thể single-clone.
#   "thivux/phoaudiobook"    : 1494h, GATED.
#   "ntt123/viet-tts-dataset": single-speaker nhưng **đóng gói trong tar.gz** — KHÔNG load qua HF
#                              datasets API thông thường. Nếu muốn dùng, cần code riêng tải tar.
DATASET_NAME      = "doof-ferb/LSVSC"
DATASET_SPLIT     = "train"
DATASET_TEXT_COL  = "transcription"   # auto-detect nếu không khớp
DATASET_AUDIO_COL = "audio"

# --- Optional pseudo-speaker filter ---
# LSVSC không có cột speaker thật nhưng có 'gender' / 'dialect' / 'emotion'. Có thể dùng làm pseudo-speaker
# để giọng nhất quán hơn. Mặc định AUTO_PSEUDO_SPEAKER=True sẽ tự pick value đông nhất từ 'gender'.
# Override: đặt PSEUDO_SPEAKER_FILTER = {"gender": "male", "dialect": "northern"} để tự chọn,
# hoặc đặt AUTO_PSEUDO_SPEAKER=False + PSEUDO_SPEAKER_FILTER=None để giữ tất cả.
PSEUDO_SPEAKER_FILTER = None
AUTO_PSEUDO_SPEAKER = True
AUTO_PSEUDO_SPEAKER_COLS = ("gender", "dialect", "emotion")

# --- Speaker filter (chỉ cho dataset CÓ cột speaker thật, vd: capleaf/viVoice) ---
# - SINGLE_SPEAKER=True : đếm speaker, giữ speaker đông nhất → single-voice clone chuẩn
# - SINGLE_SPEAKER=False: giữ tất cả → model học "voice trung bình" (default cho LSVSC)
# - SINGLE_SPEAKER=True nhưng không có cột speaker → notebook RAISE rõ ràng (không silent).
SINGLE_SPEAKER    = False
SPEAKER_COL       = None     # None => auto-detect (speaker_id, speaker, client_id, ...)

# --- Sub-sample ---
MAX_SAMPLES       = 3000     # 2000-5000 đủ cho finetune nhanh khi đã warmstart vi_VN
MIN_DURATION_S    = 1.0
MAX_DURATION_S    = 12.0
REBUILD_DATA      = False    # True => xoá AUDIO_DIR + CACHE_DIR trước khi rebuild

# --- Voice config ---
VOICE_NAME        = "vi_VN-finetune-medium"
ESPEAK_VOICE      = "vi"      # tên voice espeak-ng (verified: espeak-ng --voices=vi)
SAMPLE_RATE       = 22050     # match warmstart medium ckpt; librosa tự resample từ source

# --- Training ---
BATCH_SIZE        = 12        # T4 14.5GB usable + fp16: 24 OOM thực tế (collate pad theo clip dài nhất, MAX_DURATION_S=12s) → 12 an toàn
NUM_WORKERS       = 4         # số CPU Kaggle
ADDITIONAL_STEPS  = 8000      # số step train THÊM kể từ global_step của base ckpt
VAL_CHECK_INTERVAL = 1000
PRECISION         = "16-mixed"   # T4 dùng 16-mixed; A100/L4/H100 đổi "bf16-mixed"
LEARNING_RATE     = 2e-4
NUM_DEVICES       = None      # None => auto-detect; ép số bằng 1 hoặc 2 nếu cần
STRATEGY          = "ddp_find_unused_parameters_true"  # bắt buộc cho VITS manual-opt

# --- Paths ---
WORK_DIR          = "/kaggle/working"
DATA_DIR          = f"{WORK_DIR}/data"
AUDIO_DIR         = f"{DATA_DIR}/wavs"
CSV_PATH          = f"{DATA_DIR}/metadata.csv"
CACHE_DIR         = f"{DATA_DIR}/cache"
CKPT_DIR          = f"{WORK_DIR}/checkpoints"
CONFIG_PATH       = f"{CKPT_DIR}/{VOICE_NAME}.json"
BASE_CKPT_LOCAL   = f"{WORK_DIR}/base_vi_VN.ckpt"

# --- HF Token (cho dataset gated) ---
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Kaggle Secret")
except Exception:
    print("ℹ️  No HF_TOKEN (chỉ cần cho dataset gated)")

import os, shutil
for d in (DATA_DIR, AUDIO_DIR, CACHE_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

# Disk-space guard
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print(f"Free disk on {WORK_DIR}: {free_gb:.1f} GB")
assert free_gb > 12, (
    f"Cần >12 GB free trên {WORK_DIR}. Hiện chỉ còn {free_gb:.1f} GB. "
    "Giảm MAX_SAMPLES, bật REBUILD_DATA để wipe cache cũ, hoặc dọn dẹp Kaggle output."
)

print(f"DATASET          : {DATASET_NAME}")
print(f"MAX_SAMPLES      : {MAX_SAMPLES}")
print(f"BATCH_SIZE       : {BATCH_SIZE}  (x{NUM_DEVICES or 'auto'} GPU)")
print(f"ADDITIONAL_STEPS : {ADDITIONAL_STEPS}")
print(f"PRECISION        : {PRECISION}")


## 5. Tải base checkpoint tiếng Việt từ HuggingFace

File ~846 MB, có resume nếu bị ngắt.


In [ ]:
import os, time, pathlib

dst = pathlib.Path(BASE_CKPT_LOCAL)
if dst.exists() and dst.stat().st_size > 800_000_000:
    print(f"✅ Đã có {dst} ({dst.stat().st_size/1e6:.1f} MB)")
else:
    print("Downloading base ckpt từ HuggingFace...")
    t0 = time.time()
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(
        repo_id="rhasspy/piper-checkpoints",
        repo_type="dataset",
        filename="vi/vi_VN/vais1000/medium/epoch=4769-step=919580.ckpt",
        local_dir=WORK_DIR,
    )
    if pathlib.Path(p) != dst:
        os.replace(p, dst)
    print(f"✅ Saved {dst} ({dst.stat().st_size/1e6:.1f} MB) in {time.time()-t0:.1f}s")

# =============================================================================
# Sanitize ckpt cho PyTorch 2.6+ (torch.load weights_only=True)
# =============================================================================
# Lightning CLI (_parse_ckpt_path, cli.py) + resume load đều dùng weights_only=True.
# Ckpt cũ từ rhasspy/piper-checkpoints (train bằng old piper + PL 1.x) chứa:
#   1. pathlib.PosixPath trong hyper_parameters (key `dataset`) → UnpicklingError
#   2. hparams cũ (sample_bytes, channels, num_workers, seed, ...) không có trong
#      VitsModel của piper1-gpl → jsonargparse "does not accept option" → SystemExit
# Fix: load 1 lần với weights_only=False (nguồn tin cậy: rhasspy official trên HF),
# deep-convert về plain types, giữ CHỈ hparams kiến trúc, re-save.
#
# Lưu ý: hparams trong ckpt OVERRIDE CLI args khi parse → chỉ giữ arch params
# (resblock, upsample_*, ...) để model build đúng medium + state_dict khớp;
# training params (learning_rate, ...) bỏ → knobs của notebook có hiệu lực.
# batch_size/num_symbols là link targets (lấy từ data.*) → bỏ luôn.
import torch

_ARCH_HPARAMS = {
    "sample_rate", "num_speakers",
    "resblock", "resblock_kernel_sizes", "resblock_dilation_sizes",
    "upsample_rates", "upsample_initial_channel", "upsample_kernel_sizes",
    "filter_length", "hop_length", "win_length",
    "mel_channels", "mel_fmin", "mel_fmax",
    "inter_channels", "hidden_channels", "filter_channels",
    "n_heads", "n_layers", "kernel_size", "p_dropout", "n_layers_q",
    "use_spectral_norm", "gin_channels", "use_sdp", "segment_size",
}

def _to_plain(o):
    """Convert mọi object về types mà torch.load(weights_only=True) chấp nhận."""
    if o is None or isinstance(o, torch.Tensor):
        return o
    if type(o) in (str, int, float, bool, bytes, complex):  # exact types (np.float64 là float subclass!)
        return o
    if isinstance(o, pathlib.PurePath):
        return str(o)
    if isinstance(o, dict):  # gồm AttributeDict / OrderedDict
        return {_to_plain(k): _to_plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        t = type(o) if type(o) in (list, tuple, set) else list
        return t(_to_plain(x) for x in o)
    for base in (bool, int, float, complex, str, bytes):  # scalar subclasses: np.float64, IntEnum, ...
        if isinstance(o, base):
            return base(o)
    if hasattr(o, "item") and callable(o.item) and getattr(o, "ndim", None) == 0:
        return o.item()   # 0-d numpy scalar (np.int64, ...)
    if hasattr(o, "tolist") and callable(o.tolist):
        return o.tolist() # numpy array
    return str(o)         # object lạ → string

try:
    torch.load(dst, weights_only=True, map_location="cpu")
    print("✅ ckpt đã weights_only-safe — không cần sanitize")
except Exception as _e:
    print(f"ℹ️  Sanitizing ckpt cho PyTorch 2.6+ ({type(_e).__name__}) ...")
    t0 = time.time()
    ckpt = torch.load(dst, map_location="cpu", weights_only=False)  # trusted: rhasspy official
    ckpt = _to_plain(ckpt)
    hp = ckpt.get("hyper_parameters") or {}
    dropped = sorted(set(hp) - _ARCH_HPARAMS)
    ckpt["hyper_parameters"] = {k: v for k, v in hp.items() if k in _ARCH_HPARAMS}
    if dropped:
        print(f"   dropped hparams (không có trong VitsModel mới / để CLI thắng): {dropped}")
    _tmp = str(dst) + ".tmp"
    torch.save(ckpt, _tmp)
    os.replace(_tmp, dst)
    torch.load(dst, weights_only=True, map_location="cpu")  # verify
    print(f"✅ Sanitized + verified in {time.time()-t0:.1f}s ({dst.stat().st_size/1e6:.1f} MB)")


## 6. Tải & tiền xử lý dataset → CSV + WAV files

- Stream dataset từ HF (không tải hết về RAM)
- Auto-detect cột text + cột speaker (nếu có)
- Single-speaker filter (raise nếu bật mà không có cột speaker)
- Lọc duration, ghi WAV mono 22050Hz PCM, sinh CSV `wav|text`


In [ ]:
import os, csv, time, collections, shutil
import numpy as np
import soundfile as sf
from datasets import load_dataset, Audio

t0 = time.time()

if REBUILD_DATA:
    print("REBUILD_DATA=True → xoá AUDIO_DIR + CACHE_DIR")
    shutil.rmtree(AUDIO_DIR, ignore_errors=True)
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    os.makedirs(AUDIO_DIR, exist_ok=True)
    os.makedirs(CACHE_DIR, exist_ok=True)

def stream_ds(decode_audio: bool = True):
    # Stream dataset. decode_audio=False -> bỏ qua decode (nhanh hơn nhiều khi chỉ cần metadata).
    kw = dict(split=DATASET_SPLIT, streaming=True)
    if HF_TOKEN:
        kw["token"] = HF_TOKEN
    d = load_dataset(DATASET_NAME, **kw)
    if decode_audio:
        d = d.cast_column(DATASET_AUDIO_COL, Audio(sampling_rate=SAMPLE_RATE, mono=True))
    else:
        # Disable audio decoding → rất nhanh khi scan metadata
        d = d.cast_column(DATASET_AUDIO_COL, Audio(decode=False))
    return d

# Peek columns từ stream metadata-only (không decode audio → nhanh)
_peek_ds = stream_ds(decode_audio=False)
peek = next(iter(_peek_ds))
del _peek_ds
ds = stream_ds()  # full stream cho main loop
columns = list(peek.keys())
print("Dataset columns:", columns)

# Auto-detect text column
text_col = DATASET_TEXT_COL if DATASET_TEXT_COL in columns else None
if text_col is None:
    for c in ("transcription", "text", "sentence", "transcript", "normalized_text", "raw_transcription"):
        if c in columns:
            text_col = c; break
if text_col is None:
    raise RuntimeError(f"Không tìm thấy cột text trong: {columns}")
print("Using text column:", text_col)

# Auto-detect speaker column
speaker_candidates = [SPEAKER_COL, "speaker_id", "speaker", "speaker_name", "client_id", "reader"]
speaker_col = next((c for c in speaker_candidates if c and c in columns), None)

if SINGLE_SPEAKER:
    if speaker_col is None:
        raise RuntimeError(
            f"SINGLE_SPEAKER=True nhưng dataset {DATASET_NAME} không có cột speaker (columns={columns}). "
            "Hoặc dataset thực sự là single-speaker (như ntt123/viet-tts-dataset) thì set SINGLE_SPEAKER=False và bỏ qua filter, "
            "hoặc đổi DATASET_NAME sang dataset có speaker IDs."
        )
    print(f"Speaker column detected: {speaker_col}")
else:
    print("SINGLE_SPEAKER=False → giữ tất cả rows")

# Single-speaker filter: scan để chọn speaker đông nhất
chosen_speaker = None
if SINGLE_SPEAKER and speaker_col:
    counts = collections.Counter()
    ds_scan = stream_ds(decode_audio=False)  # metadata-only → fast
    for i, row in enumerate(ds_scan):
        sp = row.get(speaker_col)
        if sp is not None:
            counts[str(sp)] += 1
        if i >= MAX_SAMPLES * 10:
            break
    if not counts:
        raise RuntimeError(f"Không tìm thấy speaker nào trong {MAX_SAMPLES*10} rows đầu")
    chosen_speaker, n_for_chosen = counts.most_common(1)[0]
    print(f"Chosen speaker: {chosen_speaker} ({n_for_chosen} samples in scan window)")
    ds = stream_ds()  # fresh stream cho main loop

# Auto pseudo-speaker: nếu filter None + dataset có cột pseudo (gender/dialect), tự pick value đông nhất
# → voice consistency cao hơn nhiều so với toàn bộ mixed-speaker dataset.
if PSEUDO_SPEAKER_FILTER is None and AUTO_PSEUDO_SPEAKER and not SINGLE_SPEAKER:
    for auto_col in AUTO_PSEUDO_SPEAKER_COLS:
        if auto_col in columns:
            _counts = collections.Counter()
            _scan = stream_ds(decode_audio=False)  # metadata-only → fast
            for _i, _row in enumerate(_scan):
                _v = _row.get(auto_col)
                if _v is not None:
                    _counts[str(_v)] += 1
                if _i >= 800:
                    break
            if _counts:
                _top_v, _top_n = _counts.most_common(1)[0]
                PSEUDO_SPEAKER_FILTER = {auto_col: _top_v}
                print(f"Auto pseudo-speaker filter: {PSEUDO_SPEAKER_FILTER}  "
                      f"(top {_top_n}/{sum(_counts.values())} in scan; distribution: {dict(_counts.most_common(5))})")
                ds = stream_ds()
                break

# Validate PSEUDO_SPEAKER_FILTER keys exist in columns
if PSEUDO_SPEAKER_FILTER:
    missing = [k for k in PSEUDO_SPEAKER_FILTER if k not in columns]
    if missing:
        raise RuntimeError(
            f"PSEUDO_SPEAKER_FILTER keys {missing} không có trong dataset columns {columns}. "
            "Xóa key không tồn tại hoặc set PSEUDO_SPEAKER_FILTER=None."
        )
    print(f"Pseudo-speaker filter ACTIVE: {PSEUDO_SPEAKER_FILTER}")
else:
    print("Pseudo-speaker filter: None (giữ tất cả rows)")

# Main loop: ghi WAV + CSV
n_written, n_skip = 0, 0
csv_rows = []

for row in ds:
    if n_written >= MAX_SAMPLES:
        break
    txt = (row.get(text_col) or "").strip()
    if not txt:
        n_skip += 1; continue
    if chosen_speaker is not None and str(row.get(speaker_col)) != chosen_speaker:
        continue
    if PSEUDO_SPEAKER_FILTER:
        if any(str(row.get(k)) != str(v) for k, v in PSEUDO_SPEAKER_FILTER.items()):
            continue

    aud = row[DATASET_AUDIO_COL]
    arr = np.asarray(aud["array"], dtype=np.float32)
    sr  = int(aud["sampling_rate"])
    dur = len(arr) / sr
    if dur < MIN_DURATION_S or dur > MAX_DURATION_S:
        n_skip += 1; continue

    wav_name = f"vi_{n_written:06d}.wav"
    sf.write(os.path.join(AUDIO_DIR, wav_name), arr, sr, subtype="PCM_16")
    csv_rows.append((wav_name, txt))
    n_written += 1
    if n_written % 500 == 0:
        print(f"  {n_written}/{MAX_SAMPLES} ... (skip {n_skip})  elapsed {time.time()-t0:.1f}s")

# Ghi CSV với delimiter '|', strip newlines/pipes trong text
with open(CSV_PATH, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f, delimiter="|", quoting=csv.QUOTE_NONE, escapechar="\\")
    for name, text in csv_rows:
        clean = text.replace("|", " ").replace("\n", " ").replace("\r", " ").strip()
        w.writerow([name, clean])

print(f"✅ Wrote {n_written} samples in {time.time()-t0:.1f}s")
print(f"   audio_dir = {AUDIO_DIR}")
print(f"   csv_path  = {CSV_PATH}")
!head -3 "$CSV_PATH"
!ls "$AUDIO_DIR" | wc -l
!du -sh "$AUDIO_DIR"


## 7. Pre-cache phoneme + spectrogram

`prepare_data()` của Piper sẽ:
- Phonemize qua espeak-ng (`vi`)
- Trim silence bằng Silero VAD
- Pre-compute spectrogram + lưu cache `.pt`

Chạy riêng để bước train không nghẽn ở data-prep (vốn single-thread). Subprocess train sau sẽ re-iterate cache (nhanh, vì các file đã tồn tại sẽ bị skip).


In [ ]:
import sys, time, os, subprocess
sys.path.insert(0, "/kaggle/working/piper1-gpl/src")

# Self-heal: nếu kernel restart hoặc editable install chưa build extension,
# build lại trước khi import từ src/.
repo_dir = "/kaggle/working/piper1-gpl"
try:
    import piper.espeakbridge  # type: ignore[attr-defined]
except Exception:
    print("ℹ️  Building piper.espeakbridge in-place...")
    subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=repo_dir,
        check=True,
    )
    import piper.espeakbridge  # type: ignore[attr-defined]

# Self-heal: transformers (Kaggle preinstall, đòi huggingface_hub>=1) conflict với pin hub<1
# → lightning import chết qua torchmetrics (guard _TRANSFORMERS_GREATER_EQUAL_4_4).
# Piper train KHÔNG cần transformers → gỡ hẳn. Fallback rm -rf nếu pip uninstall fail.
import importlib, importlib.util, pathlib, shutil

def _purge_transformers():
    r = subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "transformers"],
        capture_output=True, text=True,
    )
    print((r.stdout or r.stderr).strip()[-300:])
    spec = importlib.util.find_spec("transformers")
    if spec is not None and spec.origin:
        pkg_dir = pathlib.Path(spec.origin).parent
        print(f"⚠️  pip uninstall không gỡ được → rm -rf {pkg_dir}")
        shutil.rmtree(pkg_dir, ignore_errors=True)
        for di in pkg_dir.parent.glob("transformers-*.dist-info"):
            shutil.rmtree(di, ignore_errors=True)
    # Purge module cache cả chain để guard re-evaluate
    for m in [k for k in sys.modules
              if k.split(".")[0] in ("lightning", "torchmetrics", "transformers",
                                     "lightning_utilities", "pytorch_lightning")]:
        del sys.modules[m]
    importlib.invalidate_caches()
    assert importlib.util.find_spec("transformers") is None, "transformers vẫn còn — restart kernel rồi chạy lại"

try:
    import lightning  # noqa: F401
except ImportError:
    print("ℹ️  lightning import failed → gỡ transformers + retry...")
    _purge_transformers()
    import lightning  # noqa: F401
print("✅ lightning import OK")

# Reset module cache để pick up patched dataset.py
for m in [k for k in sys.modules if k.startswith("piper")]:
    del sys.modules[m]

from piper.train.vits.dataset import VitsDataModule

t0 = time.time()
dm = VitsDataModule(
    csv_path=CSV_PATH,
    audio_dir=AUDIO_DIR,
    cache_dir=CACHE_DIR,
    espeak_voice=ESPEAK_VOICE,
    config_path=CONFIG_PATH,
    voice_name=VOICE_NAME,
    sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
dm.prepare_data()
print(f"✅ prepare_data() done in {time.time()-t0:.1f}s")
print(f"   cache_dir = {CACHE_DIR}")
!ls "$CACHE_DIR" | head -5
!echo -n "Total cache files: "; ls "$CACHE_DIR" | wc -l


## 8. 🏋️ Train

`python -m piper.train fit` với tất cả flag tối ưu.

### ⚠️ Quan trọng: `--ckpt_path` semantics

Lightning's `--ckpt_path` **resume** training, restore cả `global_step` từ base ckpt. Base ckpt có `global_step=919580`. Nếu set `--trainer.max_steps=8000` thì Lightning sẽ thấy `919580 > 8000` và DỪNG NGAY.

→ Notebook tự đọc `global_step` từ base ckpt rồi tính `target_max_steps = base_step + ADDITIONAL_STEPS`.


In [ ]:
import torch, os, shlex, sys, pathlib, time

# =============================================================================
# Guard: sanitize base ckpt cho PyTorch 2.6+ (weights_only=True) — idempotent.
# Đặt ngay trong cell Train để KHÔNG THỂ skip (xem chi tiết ở cell download ckpt).
# =============================================================================
_ARCH_HPARAMS = {
    "sample_rate", "num_speakers",
    "resblock", "resblock_kernel_sizes", "resblock_dilation_sizes",
    "upsample_rates", "upsample_initial_channel", "upsample_kernel_sizes",
    "filter_length", "hop_length", "win_length",
    "mel_channels", "mel_fmin", "mel_fmax",
    "inter_channels", "hidden_channels", "filter_channels",
    "n_heads", "n_layers", "kernel_size", "p_dropout", "n_layers_q",
    "use_spectral_norm", "gin_channels", "use_sdp", "segment_size",
}

def _to_plain(o):
    if o is None or isinstance(o, torch.Tensor):
        return o
    if type(o) in (str, int, float, bool, bytes, complex):
        return o
    if isinstance(o, pathlib.PurePath):
        return str(o)
    if isinstance(o, dict):
        return {_to_plain(k): _to_plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        t = type(o) if type(o) in (list, tuple, set) else list
        return t(_to_plain(x) for x in o)
    for base in (bool, int, float, complex, str, bytes):
        if isinstance(o, base):
            return base(o)
    if hasattr(o, "item") and callable(o.item) and getattr(o, "ndim", None) == 0:
        return o.item()
    if hasattr(o, "tolist") and callable(o.tolist):
        return o.tolist()
    return str(o)

try:
    torch.load(BASE_CKPT_LOCAL, weights_only=True, map_location="cpu")
    print("✅ base ckpt weights_only-safe")
except Exception as _e:
    print(f"ℹ️  Sanitizing base ckpt cho PyTorch 2.6+ ({type(_e).__name__}) ...")
    _t0 = time.time()
    _ckpt = torch.load(BASE_CKPT_LOCAL, map_location="cpu", weights_only=False)  # trusted: rhasspy official
    _ckpt = _to_plain(_ckpt)
    _hp = _ckpt.get("hyper_parameters") or {}
    _dropped = sorted(set(_hp) - _ARCH_HPARAMS)
    _ckpt["hyper_parameters"] = {k: v for k, v in _hp.items() if k in _ARCH_HPARAMS}
    if _dropped:
        print(f"   dropped hparams: {_dropped}")
    _tmp = str(BASE_CKPT_LOCAL) + ".tmp"
    torch.save(_ckpt, _tmp)
    os.replace(_tmp, BASE_CKPT_LOCAL)
    del _ckpt
    torch.load(BASE_CKPT_LOCAL, weights_only=True, map_location="cpu")  # verify
    print(f"✅ Sanitized + verified in {time.time()-_t0:.1f}s")

# =============================================================================
# Auto-resume: có ckpt finetune trong lightning_logs → train tiếp từ đó;
# chưa có → bắt đầu từ base ckpt.
# Lưu ý: PL2 migration RESET global_step về 0 khi resume từ ckpt PL1 (base),
# nên max_steps phải tính từ global_step của CHÍNH ckpt sẽ resume.
# =============================================================================
import glob
_lit_ckpts = sorted(
    glob.glob(f"{CKPT_DIR}/lightning_logs/*/checkpoints/*.ckpt"),
    key=os.path.getmtime,
)
RESUME_CKPT = _lit_ckpts[-1] if _lit_ckpts else BASE_CKPT_LOCAL
print(f"Resume from: {RESUME_CKPT}")

_meta = torch.load(RESUME_CKPT, map_location="cpu", weights_only=True)
BASE_GLOBAL_STEP = int(_meta.get("global_step", 0))
del _meta
target_max_steps = BASE_GLOBAL_STEP + ADDITIONAL_STEPS
print(f"Resume global_step  : {BASE_GLOBAL_STEP}")
print(f"Additional steps    : {ADDITIONAL_STEPS}")
print(f"Target max_steps    : {target_max_steps}")

n_gpu = torch.cuda.device_count()
devices = NUM_DEVICES or max(1, n_gpu)
print(f"Using {devices} GPU(s)")

# Env tăng tốc
env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"]      = "1"
env["TF_CPP_MIN_LOG_LEVEL"]  = "2"
env["TORCH_CUDNN_V8_API_ENABLED"] = "1"
# Giảm fragmentation (OOM log: 1.23GiB reserved-unallocated)
env["PYTORCH_ALLOC_CONF"]      = "expandable_segments:True"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # tên cũ, torch<2.9 fallback

strategy_arg = STRATEGY if devices > 1 else "auto"

cmd = [
    sys.executable, "-m", "piper.train", "fit",
    # data
    f"--data.csv_path={CSV_PATH}",
    f"--data.audio_dir={AUDIO_DIR}",
    f"--data.cache_dir={CACHE_DIR}",
    f"--data.config_path={CONFIG_PATH}",
    f"--data.voice_name={VOICE_NAME}",
    f"--data.espeak_voice={ESPEAK_VOICE}",
    f"--data.batch_size={BATCH_SIZE}",
    f"--data.num_workers={NUM_WORKERS}",
    f"--data.validation_split=0.02",
    f"--data.num_test_examples=2",
    # model
    f"--model.sample_rate={SAMPLE_RATE}",
    f"--model.learning_rate={LEARNING_RATE}",
    # trainer
    f"--trainer.max_steps={target_max_steps}",
    f"--trainer.precision={PRECISION}",
    f"--trainer.accelerator=gpu",
    f"--trainer.devices={devices}",
    f"--trainer.strategy={strategy_arg}",
    f"--trainer.benchmark=true",
    f"--trainer.val_check_interval={VAL_CHECK_INTERVAL}",
    # Dataset nhỏ → batches/epoch < VAL_CHECK_INTERVAL. check_val_every_n_epoch=null
    # đổi nghĩa val_check_interval thành GLOBAL steps (xuyên epochs) → hợp lệ.
    f"--trainer.check_val_every_n_epoch=null",
    f"--trainer.limit_val_batches=4",
    f"--trainer.log_every_n_steps=25",
    f"--trainer.default_root_dir={CKPT_DIR}",
    f"--trainer.enable_progress_bar=true",
    # resume từ checkpoint Việt (Lightning restore weights + optimizer + global_step)
    f"--ckpt_path={RESUME_CKPT}",
]
print("\nCommand:\n  " + " \\\n  ".join(shlex.quote(c) for c in cmd))


In [ ]:
import subprocess
from pathlib import Path

repo_dir = "/kaggle/working/piper1-gpl"
train_log = Path(CKPT_DIR) / "train.log"
train_log.parent.mkdir(parents=True, exist_ok=True)


def run_and_log(args):
    with open(train_log, "a", encoding="utf-8") as log_f:
        log_f.write("\n" + "=" * 80 + "\n")
        log_f.write("COMMAND:\n")
        log_f.write(" ".join(args) + "\n")
        log_f.write("=" * 80 + "\n")
        log_f.flush()
        proc = subprocess.Popen(
            args,
            cwd=repo_dir,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log_f.write(line)
        return proc.wait()


retcode = run_and_log(cmd)
print("\nTrain exit code:", retcode)

# Auto-fallback: nếu DDP 2 GPU fail → retry với 1 GPU
if retcode != 0 and devices > 1:
    print(f"\n⚠️  DDP {devices}-GPU training failed (exit {retcode}). Retrying với 1 GPU + strategy=auto...")
    cmd_1gpu = []
    for arg in cmd:
        if arg.startswith("--trainer.devices="):
            cmd_1gpu.append("--trainer.devices=1")
        elif arg.startswith("--trainer.strategy="):
            cmd_1gpu.append("--trainer.strategy=auto")
        else:
            cmd_1gpu.append(arg)
    retcode = run_and_log(cmd_1gpu)
    print("\nTrain (1-GPU retry) exit code:", retcode)

if retcode != 0:
    tail_lines = train_log.read_text(encoding="utf-8", errors="ignore").splitlines()[-120:]
    print("\n===== train.log tail =====")
    print("\n".join(tail_lines))
    print("===== end tail =====")

    tail_text = "\n".join(tail_lines)
    hints = []
    if "CUDA out of memory" in tail_text:
        hints.append("CUDA OOM: giảm BATCH_SIZE xuống 16 hoặc 12, hoặc ép NUM_DEVICES=1.")
    if "global_step" in tail_text and "max_steps" in tail_text:
        hints.append("Checkpoint resume mismatch: tăng ADDITIONAL_STEPS để target max_steps > base global_step.")
    if "size mismatch" in tail_text or "shape mismatch" in tail_text:
        hints.append("Checkpoint/model mismatch: nếu dataset thành multi-speaker thì không dùng --ckpt_path base single-speaker; chuyển sang --model.vocoder_warmstart_ckpt.")
    if "No supported gpu backend found" in tail_text or "MisconfigurationException" in tail_text:
        hints.append("Trainer/GPU config lỗi: thử NUM_DEVICES=1 và PRECISION='16-mixed'.")
    if "ImportError" in tail_text or "ModuleNotFoundError" in tail_text:
        hints.append("Môi trường chưa đủ dependency: chạy lại cell cài đặt và prepare_data từ đầu.")

    if hints:
        print("\nPossible fixes:")
        for hint in hints:
            print(f"- {hint}")

    raise RuntimeError(
        f"Training failed. Xem log đầy đủ tại: {train_log}"
    )


## 9. Tìm checkpoint mới nhất + Export ONNX


In [ ]:
import glob, os, subprocess, sys

# Tìm tất cả ckpt do Lightning ghi, lấy mới nhất theo mtime
patterns = [
    f"{CKPT_DIR}/**/checkpoints/*.ckpt",
    f"{CKPT_DIR}/**/*.ckpt",
]
ckpts = set()
for pat in patterns:
    ckpts.update(glob.glob(pat, recursive=True))
ckpts = sorted(ckpts, key=os.path.getmtime)
# Loại trừ base ckpt (nếu lỡ nằm trong CKPT_DIR)
ckpts = [c for c in ckpts if os.path.abspath(c) != os.path.abspath(BASE_CKPT_LOCAL)]
assert ckpts, f"Không tìm thấy checkpoint mới trong {CKPT_DIR}. Lightning có thể chưa lưu (cần ít nhất 1 validation pass)."
latest = ckpts[-1]
print(f"Latest checkpoint: {latest}  ({os.path.getsize(latest)/1e6:.1f} MB)")

# -----------------------------------------------------------------------------
# Patch export_onnx.py: torch >= 2.9 đổi default torch.onnx.export sang dynamo=True
# → import onnxscript (không có trên Kaggle) + dynamo trace VITS dễ chết.
# Ép dynamo=False → dùng legacy TorchScript exporter (chỉ cần onnx, đã pin ở deps).
# -----------------------------------------------------------------------------
from pathlib import Path
_exp = Path("/kaggle/working/piper1-gpl/src/piper/train/export_onnx.py")
_src = _exp.read_text()
if "dynamo=False" not in _src:
    _anchor = "        verbose=False,\n"
    assert _anchor in _src, "export_onnx.py anchor không khớp — upstream đổi?"
    _src = _src.replace(_anchor, _anchor + "        dynamo=False,  # legacy exporter: torch>=2.9 default dynamo=True cần onnxscript\n")
    compile(_src, str(_exp), "exec")
    _exp.write_text(_src)
    print("✅ Patched export_onnx.py: dynamo=False")
else:
    print("ℹ️  export_onnx.py đã patch")

onnx_out = f"{CKPT_DIR}/{VOICE_NAME}.onnx"
ret = subprocess.run([
    sys.executable, "-m", "piper.train.export_onnx",
    "--checkpoint", latest,
    "--output-file", onnx_out,
], cwd="/kaggle/working/piper1-gpl")
assert ret.returncode == 0, "ONNX export failed!"

# Copy voice config (sinh ra ở prepare_data)
import shutil
shutil.copy(CONFIG_PATH, onnx_out + ".json")

print("\n✅ Exported:")
print(f"   {onnx_out}        ({os.path.getsize(onnx_out)/1e6:.1f} MB)")
print(f"   {onnx_out}.json")


## 10. ✅ Test inference + nghe thử


In [ ]:
import numpy as np, soundfile as sf, IPython.display as ipd
from piper import PiperVoice

voice = PiperVoice.load(onnx_out, config_path=onnx_out + ".json")
text = "Xin chào, đây là giọng nói tiếng Việt được tinh chỉnh trên Kaggle bằng Piper TTS."

audio_chunks = []
for chunk in voice.synthesize(text):
    arr = getattr(chunk, "audio_float_array", None)
    if arr is None:
        arr = chunk.audio_int16_array.astype(np.float32) / 32768.0
    audio_chunks.append(arr)
audio = np.concatenate(audio_chunks)
sr = voice.config.sample_rate

out_wav = f"{WORK_DIR}/demo_vi.wav"
sf.write(out_wav, audio, sr, subtype="PCM_16")
print(f"✅ Saved {out_wav}  duration={len(audio)/sr:.2f}s")
ipd.display(ipd.Audio(audio, rate=sr))


## 📦 Output files

Sau khi chạy xong, các file quan trọng tại `/kaggle/working/`:

| File | Mô tả |
|---|---|
| `checkpoints/<VOICE_NAME>.onnx` | Model ONNX cho inference |
| `checkpoints/<VOICE_NAME>.onnx.json` | Config (sample_rate, phoneme_id_map, ...) |
| `checkpoints/lightning_logs/version_*/checkpoints/*.ckpt` | PyTorch Lightning ckpt (để resume train) |
| `demo_vi.wav` | Audio demo |
| `checkpoints/lightning_logs/` | TensorBoard logs |

**Tải về local:**
- Kaggle tab **Output** → click file → Download
- Hoặc nén: `!cd /kaggle/working && zip -r piper_vi.zip checkpoints demo_vi.wav`

**Để tiếp tục train ở session sau:**
1. Save notebook + Output.
2. Session mới: REBUILD_DATA=False (giữ cache), `--ckpt_path=<ckpt vừa train>` thay vì base.
3. Tăng `ADDITIONAL_STEPS` cho lần tiếp.

---

## ⚠️ Troubleshooting

| Lỗi | Cách fix |
|---|---|
| `CUDA out of memory` | Giảm `BATCH_SIZE` (24→16→12) hoặc `NUM_DEVICES=1` |
| `espeak-ng: voice 'vi' not found` | Cell 2 chưa chạy xong |
| `monotonic_align` ImportError | Cell 2 chưa build cython; chạy lại `bash build_monotonic_align.sh` |
| `404 from HF (gated dataset)` | Cần `HF_TOKEN` Kaggle secret với quyền truy cập |
| `RuntimeError: SINGLE_SPEAKER=True nhưng ...` | Set `SINGLE_SPEAKER=False` (cho dataset không có speaker col) hoặc đổi dataset |
| Training dừng ngay sau khi load ckpt | `max_steps` < `BASE_GLOBAL_STEP`. Tăng `ADDITIONAL_STEPS` |
| `state_dict shape mismatch` khi load base | Đang train multi-speaker với base single-speaker. Dùng `--model.vocoder_warmstart_ckpt` thay `--ckpt_path` |
| Quá chậm | Giảm `MAX_SAMPLES`, tăng `NUM_WORKERS`, hoặc thử `bf16-mixed` (chỉ GPU support) |

**Tham khảo:**
- [Piper TRAINING.md](https://github.com/OHF-Voice/piper1-gpl/blob/main/docs/TRAINING.md)
- [Lightning Trainer flags](https://lightning.ai/docs/pytorch/stable/common/trainer.html)
- [rhasspy/piper-checkpoints](https://huggingface.co/datasets/rhasspy/piper-checkpoints)
